# 05. So Sánh Kết Quả & Trực Quan Hóa

Notebook này:
1. Tổng hợp kết quả 3 mô hình
2. Vẽ biểu đồ so sánh
3. Phân tích và nhận xét

In [ ]:
# --- Cho phép import gói src/ (notebook đặt ở thư mục gốc dự án) ---
import sys
from pathlib import Path
_root = Path.cwd()
if not (_root / "src").is_dir() and (_root.parent / "src").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import pickle

from src import config
from src.plots import plot_comparison, plot_recall_by_k
from IPython.display import Image, display


## 5.1. Kết quả tổng hợp

Notebook này đọc kết quả **thật** từ `output/all_results.pkl`.

File này được sinh ra khi chạy `python run_all.py`, hoặc khi chạy lần lượt notebook **03** (Popularity + SKNN) và **04** (GRU4Rec) — cả hai đều ghi kết quả vào cùng file. Không có số nào được điền tay.

In [ ]:
assert config.RESULTS_PATH.exists(), (
    'Chưa có output/all_results.pkl.\n'
    'Hãy chạy `python run_all.py` HOẶC chạy notebook 03 + 04 trước.'
)
with open(config.RESULTS_PATH, 'rb') as f:
    data = pickle.load(f)

meta = data.get('meta', {})
n_epochs = meta.get('n_epochs', '?')
results = {
    'Popularity Baseline': data['popularity'],
    'SKNN (k=%s)' % meta.get('sknn_k', config.SKNN_K): data['sknn'],
    'GRU4Rec (%s epoch)' % n_epochs: data['gru4rec'],
}
print('=' * 55); print('  BẢNG KẾT QUẢ TỔNG HỢP (số thật)'); print('=' * 55)
print(f'  {"Mô hình":<25} {"Recall@20":>10} {"MRR@20":>8}')
print(f'  {"-"*45}')
for name, m in results.items():
    print(f'  {name:<25} {m["recall"]:>10.4f} {m["mrr"]:>8.4f}')
print('=' * 55)
if meta.get('n_eval'):
    print(f'  Đánh giá trên {meta["n_eval"]:,} phiên test')
if meta.get('split'):
    print(f'  Cách chia dữ liệu: {meta["split"]}')


## 5.2. Biểu đồ so sánh Recall@20

In [ ]:
# Vẽ qua src.plots (tiêu đề có dấu, đồng nhất với báo cáo) rồi hiển thị PNG
plot_comparison(data, config.OUTPUT_DIR)
Image(filename=str(config.OUTPUT_DIR / 'so_sanh_3_mo_hinh.png'))


## 5.3. Biểu đồ ảnh hưởng của K (SKNN)

In [ ]:
if data.get('k_experiment'):
    plot_recall_by_k(data, config.OUTPUT_DIR)
    for k, r in zip(data['k_experiment']['k_values'], data['k_experiment']['recalls']):
        print(f'  K={k:4d} -> Recall@20 = {r:.4f}')
    display(Image(filename=str(config.OUTPUT_DIR / 'recall_theo_k.png')))
else:
    print('Chưa có thí nghiệm K trong all_results.pkl → chạy `python run_all.py`.')


## 5.4. Phân tích và nhận xét

In [ ]:
pop_r, sknn_r, gru_r = data['popularity']['recall'], data['sknn']['recall'], data['gru4rec']['recall']
pop_m, sknn_m, gru_m = data['popularity']['mrr'], data['sknn']['mrr'], data['gru4rec']['mrr']

print('=' * 60); print('  PHÂN TÍCH KẾT QUẢ (số thực nghiệm)'); print('=' * 60)
print()
print('1. SKNN vượt Popularity Baseline:')
print(f'   Recall@20: {sknn_r:.4f} vs {pop_r:.4f}  (+{(sknn_r-pop_r)*100:.1f} điểm)')
print('   → Tận dụng thông tin chuỗi trong phiên có giá trị lớn.')
print()
diff = gru_r - sknn_r
print('2. GRU4Rec so với SKNN:')
print(f'   Recall@20: GRU={gru_r:.4f}  SKNN={sknn_r:.4f}  (chênh {diff*100:+.1f} điểm)')
if gru_r >= sknn_r:
    print('   → GRU4Rec nhỉnh hơn: học được pattern thứ tự click.')
else:
    print('   → Trên mẫu 1/64 + chia theo thời gian, SKNN vẫn nhỉnh hơn GRU4Rec.')
    print('     Lý do thường gặp: dữ liệu ít, GRU mới 10 epoch chưa hội tụ hẳn,')
    print('     và SKNN rất mạnh khi item phổ biến lặp lại giữa các phiên gần nhau.')
print()
ke = data.get('k_experiment')
if ke:
    kv, kr = ke['k_values'], ke['recalls']
    best_i = max(range(len(kr)), key=lambda i: kr[i])
    print('3. Ảnh hưởng của K (SKNN):')
    for k, r in zip(kv, kr):
        mark = '  ← tốt nhất' if k == kv[best_i] else ''
        print(f'   K={k:4d} -> Recall@20={r:.4f}{mark}')
    print(f'   → Recall tăng dần rồi bão hòa; K={kv[best_i]} cho kết quả tốt nhất.')
    print()
print(f'4. Ưu điểm SKNN: đơn giản, không cần GPU/huấn luyện, kết quả tốt (Recall@20 = {sknn_r*100:.1f}%).')
print('5. Ưu điểm GRU4Rec: học được thứ tự click; còn dư địa cải thiện (thêm epoch, dữ liệu, siêu tham số).')
print()
best_name = max([('Popularity', pop_r), ('SKNN', sknn_r), ('GRU4Rec', gru_r)], key=lambda x: x[1])[0]
print('6. Kết luận:')
print('   Session-based RS xử lý tốt bài toán khách ẩn danh.')
print(f'   Với cấu hình thí nghiệm này, mô hình tốt nhất: {best_name} (Recall@20 = {max(pop_r, sknn_r, gru_r)*100:.1f}%).')
print('   SKNN là lựa chọn thực dụng; GRU4Rec tiềm năng khi có thêm dữ liệu/epoch.')


## 5.5. Tài liệu tham khảo

1. Hidasi et al. (2016). *Session-based Recommendations with Recurrent Neural Networks*. ICLR. https://arxiv.org/abs/1511.06939
2. Ludewig & Jannach (2018). *Evaluation of Session-based Recommendation Algorithms*. UMUAI. https://arxiv.org/abs/1803.09587
3. Jannach et al. (2017). *When RNNs meet the Neighborhood for Session-Based Recommendation*. RecSys.
4. Li et al. (2017). *Neural Attentive Session-based Recommendation*. CIKM.
5. Dataset: Yoochoose - RecSys Challenge 2015. https://www.kaggle.com/datasets/chadgostopp/recsys-challenge-2015